# 08 — Modo batch y sincronización de múltiples dispositivos

Este notebook cubre patrones avanzados para coordinar múltiples dispositivos Ivium simultáneamente:

- **Parámetro de estado** — un entero global compartido entre Python e IviumSoft usado como señal de sincronización
- **Puntos de ajuste multi-dispositivo** — aplicar potencial o corriente a múltiples dispositivos en una llamada sin cambiar instancias
- **Patrones de experimentos paralelos** — iniciar y monitorear mediciones en todos los dispositivos de forma concurrente

### Casos de uso típicos

- Disparar todos los dispositivos para iniciar una medición al mismo tiempo
- Hacer que los scripts del lado de IviumSoft esperen una señal de Python antes de continuar
- Aplicar el mismo punto de ajuste a múltiples dispositivos simultáneamente
- Ejecutar experimentos independientes en paralelo y recopilar todos los resultados

### Requisitos previos
- Múltiples instancias de IviumSoft en ejecución (o un Ivium-n-Soft con múltiples canales)
- Al menos dos dispositivos conectados para las secciones multi-dispositivo
- Referencia de manejo de errores: `01_getting_started.ipynb` — Sección 4

In [ ]:
import time
import pandas as pd
import matplotlib.pyplot as plt
from pyvium import Pyvium

Pyvium.open_driver()
print(f"Active instances: {Pyvium.get_active_iviumsoft_instances()}")

## Multicanal vs Modo MC — Elegir la arquitectura correcta

IviumSoft proporciona dos modos multi-dispositivo distintos. Elige según el número de canales:

| Característica | **Multicanal** | **Modo MC** |
|---------|-----------------|-------------|
| Canales máximos | **32** | **128** |
| Visualización en tiempo real | Sí — trazas de corriente/potencial en vivo | No |
| Acceso a datos durante la ejecución | `get_available_data_points_number()` funciona | No accesible hasta completar |
| Control de IviumSoft | `select_channel(n)` (dentro de una ventana) | Instancias separadas de IviumSoft o con scripts |
| Caso de uso | ≤32 electrodos, necesita monitoreo en vivo | >32 electrodos, más importante el rendimiento |
| API de Pyvium | `select_channel(n)` por canal | `select_iviumsoft_instance(n)` por instancia |

> **Regla general:** Para ≤32 canales usa **Multicanal** (Ivium-n-Soft). Para >32 canales usa **Modo MC** con múltiples instancias de IviumSoft.

> **Nota:** El modo Multicanal usa `select_channel(n)` para cambiar entre canales dentro de una **única** ventana de IviumSoft. El Modo MC usa `select_iviumsoft_instance(n)` para direccionar ventanas separadas de IviumSoft.

## 1. El parámetro de estado

`set_status_par(value)` / `get_status_par()` exponen un entero global compartido entre Python e IviumSoft. Es por instancia — cada ventana de IviumSoft tiene su propio parámetro de estado.

**Datos clave del manual de IviumSoft:**
- StatusPar se **inicializa a `-1` al arrancar el PC**
- **Retiene su último valor establecido** hasta que el PC se reinicia — no se restablece entre mediciones o ciclos de apertura/cierre del driver
- Tanto Python (vía pyvium) como los scripts Batch de IviumSoft pueden leerlo y escribirlo

### Convención de sincronización

Elige cualquier convención que se adapte a tu flujo de trabajo. Una común es:

| Valor | Quién lo establece | Significado |
|-------|------------|----------|
| `-1` | Valor predeterminado al arrancar el PC | Sin inicializar |
| `0` | Python | Restablecer / inactivo |
| `1` | Python | "Listo — IviumSoft puede iniciar" |
| `2` | Batch de IviumSoft | "Medición en ejecución" |
| `3` | Batch de IviumSoft | "Medición completada" |
| `-99` | Cualquiera | Cancelar / error |

In [ ]:
Pyvium.select_iviumsoft_instance(1)

# Escribir un valor
Pyvium.set_status_par(0)
print(f"Set status par to 0")

# Leerlo de vuelta
val = Pyvium.get_status_par()
print(f"Get status par: {val}")

## 2. Python → IviumSoft: Señal para iniciar

Patrón: Python establece el parámetro de estado a `1` para señalar a IviumSoft que inicie un método. Los scripts Batch del lado de IviumSoft sondean el parámetro de estado y comienzan cuando ven `1`.

Esto permite que la automatización del lado de IviumSoft reaccione a los eventos de Python.

In [ ]:
# Señalar a IviumSoft que continúe
Pyvium.select_iviumsoft_instance(1)
Pyvium.set_status_par(1)  # "ir"
print("Triggered IviumSoft instance 1 (status par = 1)")

## 3. El contrato bidireccional completo

El parámetro de estado funciona en **ambas direcciones**. Los scripts Batch de IviumSoft tienen dos DirectCommands coincidentes:

| Dirección | Lado Python | Lado Batch IviumSoft |
|-----------|-------------|----------------------|
| Python → IviumSoft | `set_status_par(1)` | `WaitForStatusPar` — detiene Batch hasta que el valor coincide |
| IviumSoft → Python | `get_status_par()` en un bucle de sondeo | `SetStatusPar` — Batch establece el valor cuando termina |

### Configuración del script Batch de IviumSoft (para referencia)

Para hacer que un script Batch de IviumSoft espere la señal de inicio de Python y luego informe cuando termine, configura una línea `DirectCommand` en el Editor Batch con:

```
DirectCommand:
  WaitForStatusPar  ✓  Value = 1   Timeout = 300 s   ← espera a que Python diga "ir"
  (luego ejecuta tus métodos)
  SetStatusPar      ✓  Value = 3                      ← señala a Python "listo"
```

Python entonces conduce la secuencia:

```python
Pyvium.set_status_par(0)   # restablecer
# ... preparar experimento ...
Pyvium.set_status_par(1)   # señalar a IviumSoft que inicie
# ... esperar a que IviumSoft establezca estado = 3 ...
```

In [ ]:
def wait_for_status(instance: int, target_value: int, timeout_s: float = 120.0):
    """Bloquear hasta que el parámetro de estado de la instancia dada alcance target_value, o tiempo de espera."""
    Pyvium.select_iviumsoft_instance(instance)
    t_start = time.time()

    while True:
        current = Pyvium.get_status_par()
        if current == target_value:
            return True
        if time.time() - t_start > timeout_s:
            print(f"Timeout waiting for instance {instance} status={target_value}")
            return False
        time.sleep(0.2)

# Ejemplo: esperar hasta 60 s para que IviumSoft establezca el parámetro de estado a 3 ("listo")
# done = wait_for_status(instance=1, target_value=3, timeout_s=60.0)
# if done:
#     print("Measurement done")
print("wait_for_status() helper defined")

## 4. Disparadores alternativos — Salida digital y entrada analógica

Además de StatusPar, el Modo Batch de IviumSoft admite dos mecanismos de disparo a nivel hardware que Python puede controlar directamente:

### Disparo digital (`WaitForDigIn1`)

El Batch de IviumSoft puede detenerse en una línea `DirectCommand` configurada como:
```
WaitForDigIn1  ✓   WaitForHi = ✓   Timeout = 60 s
```
Python controla esto afirmando o liberando una línea de salida digital en el puerto externo, que está cableada de vuelta a la entrada digital 1 del dispositivo.

### Disparo analógico (`WaitForAn1`)

El Batch de IviumSoft puede detenerse hasta que la Entrada Analógica 1 supera un voltaje umbral:
```
WaitForAn1  ✓   UntilAn1 > 2.5 V   Timeout = 60 s
```
Python controla esto estableciendo el canal DAC 0 al voltaje de disparo.

Ambos mecanismos tienen un **tiempo de espera** — el script Batch continúa independientemente una vez que el tiempo de espera expira, por lo que los experimentos nunca se quedan permanentemente bloqueados por una señal fallida de Python.

In [ ]:
import time

# --- Patrón de disparo digital ---
# El Batch de IviumSoft está configurado con WaitForDigIn1 (WaitForHi, Timeout=60s)
# Python afirma la línea de salida digital 0 para liberar la espera del Batch

def trigger_batch_via_digital(line: int = 0):
    """Afirmar una línea de salida digital para disparar un Batch WaitForDigIn de IviumSoft."""
    Pyvium.set_digital_output(1 << line)   # llevar línea a ALTO
    print(f"Digital line {line} HIGH — IviumSoft Batch trigger sent")

def release_digital_trigger(line: int = 0):
    """Liberar la línea digital después de que el Batch haya visto el disparo."""
    current = Pyvium.get_digital_input()
    Pyvium.set_digital_output(0)            # todas las líneas a BAJO
    print("Digital lines reset to LOW")

# Ejemplo de uso:
# trigger_batch_via_digital(0)
# time.sleep(0.1)
# release_digital_trigger(0)

print("Digital trigger helpers defined")

# --- Patrón de disparo analógico ---
# El Batch de IviumSoft está configurado con WaitForAn1 (UntilAn1 > 2.5 V, Timeout=60s)
# Python eleva el DAC 0 por encima del umbral para liberar la espera

TRIGGER_VOLTAGE  = 3.0   # V — debe superar el umbral del Batch
IDLE_VOLTAGE     = 0.0   # V

def trigger_batch_via_analog():
    Pyvium.set_dac(0, TRIGGER_VOLTAGE)
    print(f"DAC 0 → {TRIGGER_VOLTAGE} V — IviumSoft Batch analog trigger sent")

def release_analog_trigger():
    Pyvium.set_dac(0, IDLE_VOLTAGE)
    print(f"DAC 0 → {IDLE_VOLTAGE} V (idle)")

# Ejemplo de uso:
# trigger_batch_via_analog()
# time.sleep(0.5)
# release_analog_trigger()

print("Analog trigger helpers defined")

## 5. Lanzar Python desde un script Batch de IviumSoft (`ExecuteProgram`)

El Modo Batch de IviumSoft tiene un DirectCommand `ExecuteProgram` que puede lanzar cualquier ejecutable externo — incluido un script de Python — en un punto específico de una secuencia Batch. Con `WaitUntilFinished = ✓`, el Batch se pausa hasta que el proceso Python termina.

**Configuración del Batch:**
```
DirectCommand:
  ExecuteProgram     ✓
  Program Name:      python.exe
  Path:              C:\Users\...\Scripts\
  Command Line:      C:\ruta\a\mi_analisis.py --output C:\datos\resultado.csv
  WaitUntilFinished: ✓
```

**Casos de uso:**
- Post-procesar datos inmediatamente después de cada barrido dentro de un bucle Batch
- Enviar una notificación (correo electrónico, Slack) cuando termina un experimento largo
- Leer el resultado de la medición y escribir un nuevo archivo de parámetros que la siguiente línea Batch recupera

**Flujo desde la perspectiva de IviumSoft:**
```
[Línea Batch N]    ExecuteMethod           ← ejecutar experimento
[Línea Batch N+1]  ExecuteProgram          ← lanzar Python, esperar a que termine
[Línea Batch N+2]  Volver / siguiente barrido  ← resultado de Python ya en disco
```

Esto es el inverso de los patrones anteriores: en lugar de que Python conduzca IviumSoft, IviumSoft conduce Python en el momento preciso de la secuencia.

## Batch de IviumSoft — EditMethod y unidades SI

Cuando un script Batch de IviumSoft modifica parámetros del método usando el DirectCommand **`EditMethod`**, todos los valores de parámetros deben especificarse en **unidades base SI** — independientemente de las unidades de visualización mostradas en la UI de IviumSoft.

Esto es importante cuando Python genera valores de parámetros que posteriormente lee un script Batch de IviumSoft (p.ej., escritos en un archivo que Batch recoge), o cuando estás depurando por qué un método modificado por Batch no se comporta como se espera.

| Parámetro | Unidad de visualización en IviumSoft | Unidad SI para EditMethod / Pyvium |
|-----------|--------------------------|--------------------------------|
| Potencial | mV | **V** |
| Corriente | µA, mA | **A** |
| Tiempo | ms, min, h | **s** |
| Frecuencia | kHz | **Hz** |
| Tasa de barrido | mV/s | **V/s** |

> **Esta es la misma convención usada por `set_method_parameter()` de Pyvium** — todos los valores están en unidades base SI. IviumSoft maneja la conversión de visualización de unidades internamente.

> **Trampa:** Si estableces `'scanrate'` a `'50'` esperando 50 mV/s, la tasa de barrido real será 50 V/s. Usa `'0.05'` para 50 mV/s.

## 4. Disparar todas las instancias simultáneamente

Alternando rápidamente por todas las instancias y estableciendo el parámetro de estado, puedes disparar todos los dispositivos casi simultáneamente.

In [ ]:
def broadcast_status(value: int):
    """Establecer el parámetro de estado en todas las instancias activas de IviumSoft."""
    instances = Pyvium.get_active_iviumsoft_instances()
    for inst in instances:
        Pyvium.select_iviumsoft_instance(inst)
        Pyvium.set_status_par(value)
    print(f"Broadcast status={value} to instances {instances}")

broadcast_status(0)   # restablecer todos a inactivo
time.sleep(1.0)
broadcast_status(1)   # disparar todos

## 5. Puntos de ajuste multi-dispositivo

`set_device_potential(instance, V)` y `set_device_current(instance, A)` aplican un punto de ajuste a una instancia de dispositivo específica **sin cambiar la instancia actualmente seleccionada**. Esto permite actualizaciones rápidas de puntos de ajuste en todos los dispositivos sin la sobrecarga de cambiar de instancia.

- `instance`: el número de instancia de IviumSoft (base 1)
- Ambas funciones requieren que el driver esté abierto pero **no** que el dispositivo objetivo sea la instancia actualmente seleccionada

In [ ]:
# Aplicar el mismo potencial a todas las instancias activas simultáneamente
HOLD_POTENTIAL = 0.1  # V

instances = Pyvium.get_active_iviumsoft_instances()
for inst in instances:
    Pyvium.set_device_potential(inst, HOLD_POTENTIAL)
    print(f"Instance {inst}: potential → {HOLD_POTENTIAL} V")

print("All devices set")

In [ ]:
# Aplicar un gradiente de potenciales entre dispositivos
potentials = [0.0, 0.1, 0.2, 0.3]  # V — uno por instancia

for inst, V in zip(instances, potentials):
    Pyvium.set_device_potential(inst, V)
    print(f"Instance {inst}: {V:.2f} V")

## 6. Ejecutar métodos en todos los dispositivos en paralelo

Iniciar un método en cada dispositivo, luego recopilar datos de todos después de que todos completen.

In [ ]:
METHOD_PATH     = r"C:/IviumStat/datafiles/TEST1.imf"
EXPECTED_POINTS = 160
POLL_INTERVAL   = 0.5

def start_all(instances, method_path):
    for inst in instances:
        Pyvium.select_iviumsoft_instance(inst)
        Pyvium.connect_device()
        Pyvium.start_method(method_path)
        print(f"Instance {inst}: method started")

def wait_all(instances, expected_points):
    remaining = set(instances)
    while remaining:
        finished = set()
        for inst in remaining:
            Pyvium.select_iviumsoft_instance(inst)
            n = Pyvium.get_available_data_points_number()
            if n >= expected_points:
                finished.add(inst)
        for inst in finished:
            print(f"  Instance {inst}: complete")
        remaining -= finished
        if remaining:
            time.sleep(POLL_INTERVAL)
    print("All measurements complete")

def collect_all(instances):
    results = {}
    for inst in instances:
        Pyvium.select_iviumsoft_instance(inst)
        n = Pyvium.get_available_data_points_number()
        rows = [Pyvium.get_data_point(i) for i in range(1, n + 1)]
        results[inst] = pd.DataFrame(rows, columns=["E (V)", "I (A)", "aux"])
        print(f"  Instance {inst}: {len(rows)} points collected")
    return results

print("Helpers defined. Uncomment below to run:")
# start_all(instances, METHOD_PATH)
# wait_all(instances, EXPECTED_POINTS)
# data = collect_all(instances)

## 7. Graficar resultados multi-dispositivo

In [ ]:
# Asume que el dict `data` fue poblado por collect_all() arriba
# data = collect_all(instances)

# Ejemplo sintético para ilustración
import numpy as np
data = {
    1: pd.DataFrame({"E (V)": np.linspace(0, 1, 100),
                     "I (A)": np.random.randn(100) * 1e-6 + 1e-5}),
    2: pd.DataFrame({"E (V)": np.linspace(0, 1, 100),
                     "I (A)": np.random.randn(100) * 1e-6 + 2e-5}),
}

fig, ax = plt.subplots(figsize=(8, 5))
for inst, df in data.items():
    ax.plot(df["E (V)"], df["I (A)"] * 1e6, label=f"Device {inst}")

ax.set_xlabel("Potential (V)")
ax.set_ylabel("Current (µA)")
ax.set_title("Parallel LSV — All Devices")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Flujo de trabajo sincronizado completo

Plantilla completa: conectar todos los dispositivos, ejecutar un método en paralelo, recopilar datos, guardar y graficar.

## Canales sincronizados — Regla del maestro último

Cuando se ejecutan múltiples canales en modo sincronizado (usando la función de sincronización de canales de IviumSoft), el **canal maestro siempre debe iniciarse último**. El canal maestro actúa como fuente de reloj para todos los canales sincronizados; los canales esclavos esperan en un estado listo hasta que el maestro inicia.

**Regla crítica:** Si inicias el maestro primero, la señal de sincronización se dispara antes de que cualquier canal esclavo esté escuchando — todos los esclavos pierden el disparo e inician de forma independiente (no sincronizados).

```python
# Orden correcto para inicio multicanal sincronizado
slaves  = [2, 3, 4]  # instancias de canal esclavo
master  = 1          # instancia de canal maestro

# 1. Iniciar todos los esclavos primero (entran en estado de espera)
for inst in slaves:
    Pyvium.select_iviumsoft_instance(inst)
    Pyvium.start_method(method_path)
    print(f"Slave instance {inst}: waiting for sync")

# 2. Pequeño retraso para asegurar que los esclavos estén listos
time.sleep(0.5)

# 3. Iniciar el maestro AL FINAL — dispara el trigger de sincronización
Pyvium.select_iviumsoft_instance(master)
Pyvium.start_method(method_path)
print(f"Master instance {master}: started — sync trigger fired")
```

> Esta regla aplica a la función de canal sincronizado de IviumSoft (configurada en la configuración Multicanal). Es diferente de la sincronización de nivel software con el parámetro de estado descrita en las secciones anteriores, que no es crítica en temporización.

In [ ]:
import os

METHOD_PATH    = r"C:/IviumStat/datafiles/TEST1.imf"
OUTPUT_DIR     = r"C:/IviumStat/Data"
EXPECTED_PTS   = 160

Pyvium.open_driver()
instances = Pyvium.get_active_iviumsoft_instances()
print(f"Devices: {instances}")

# Paso 1: conectar todos e iniciar
for inst in instances:
    Pyvium.select_iviumsoft_instance(inst)
    Pyvium.connect_device()

for inst in instances:
    Pyvium.select_iviumsoft_instance(inst)
    Pyvium.start_method(METHOD_PATH)
    print(f"  Instance {inst}: started")

# Paso 2: esperar a que todos terminen
remaining = set(instances)
while remaining:
    done = set()
    for inst in remaining:
        Pyvium.select_iviumsoft_instance(inst)
        if Pyvium.get_available_data_points_number() >= EXPECTED_PTS:
            done.add(inst)
    remaining -= done
    if remaining:
        time.sleep(0.5)
print("All done")

# Paso 3: recopilar y guardar
for inst in instances:
    Pyvium.select_iviumsoft_instance(inst)
    out = os.path.join(OUTPUT_DIR, f"device_{inst}.idf")
    if os.path.isdir(OUTPUT_DIR):
        Pyvium.save_data(out)
        print(f"  Instance {inst} → {out}")

Pyvium.close_driver()

---

## Resumen

### Parámetro de estado (sincronización)

| Tarea | Python | Lado Batch IviumSoft |
|------|--------|----------------------|
| Escribir estado | `set_status_par(value)` | DirectCommand `SetStatusPar` |
| Leer / esperar en estado | `get_status_par()` en bucle de sondeo | DirectCommand `WaitForStatusPar` |
| Transmitir a todas las instancias | Bucle + `select_iviumsoft_instance(n)` + `set_status_par(v)` | — |
| Valor inicial al arrancar el PC | — | **-1** (retiene hasta reinicio del PC) |

### Disparadores hardware alternativos

| Tipo de disparo | Python envía | IviumSoft Batch espera en |
|-------------|-------------|---------------------------|
| Digital | `set_digital_output(bitmask)` | `WaitForDigIn1` (HI o LO, con tiempo de espera) |
| Analógico | `set_dac(0, voltage)` | `WaitForAn1 > threshold` (con tiempo de espera) |
| Lanzar Python | — | `ExecuteProgram` (opcional `WaitUntilFinished`) |

### Puntos de ajuste multi-dispositivo (sin cambio de instancia necesario)

| Tarea | Método |
|------|-------|
| Establecer potencial en cualquier instancia | `set_device_potential(instance, V)` |
| Establecer corriente en cualquier instancia | `set_device_current(instance, A)` |

### Elección de arquitectura multicanal

| Canales | Arquitectura | API de Pyvium |
|----------|-------------|---------------|
| ≤ 32 | Multicanal (Ivium-n-Soft) — visualización en vivo | `select_channel(n)` |
| > 32 | Modo MC — sin visualización en vivo | `select_iviumsoft_instance(n)` |

### Regla de canales sincronizados

**Siempre inicia los canales esclavos antes que el canal maestro.** El maestro dispara el trigger de sincronización al iniciar — los esclavos ya deben estar esperando.

### Unidades de EditMethod / set_method_parameter

Todos los valores de parámetros están en **unidades base SI** (V, A, s, Hz) independientemente de las unidades de visualización mostradas en la UI de IviumSoft.

### Patrón de medición paralela

```python
# 1. Iniciar todos
for inst in instances:
    Pyvium.select_iviumsoft_instance(inst)
    Pyvium.start_method(path)

# 2. Esperar a todos
while not all_done(instances):
    time.sleep(0.5)

# 3. Recopilar de todos
for inst in instances:
    Pyvium.select_iviumsoft_instance(inst)
    data[inst] = read_all_points()
```

---

## Serie de notebooks completa

| Notebook | Tema | ¿Se necesita hardware? |
|----------|-------|:--:|
| 01 | Primeros pasos, manejo de errores | No (solo IviumSoft) |
| 02 | Gestión de dispositivos e instancias | Sí |
| 03 | Modo directo — fundamentos del potenciostato | Sí |
| 04 | Modo directo — trazas, DAC/ADC, E/S digital, AC | Sí |
| 05 | BiStat (WE2) y multicanal WE32 | Sí (+ accesorios) |
| 06 | Modo de método — ejecutar experimentos, leer datos | Sí |
| 07 | Parseo de archivos IDF y exportación a CSV | **No** |
| 08 | Modo batch y sincronización de múltiples dispositivos | Sí (múltiples dispositivos) |